In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
spark = SparkSession.builder.getOrCreate()
spark.range(10).show()

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
|  5|
|  6|
|  7|
|  8|
|  9|
+---+



2.2 Verify Your Connection

In [2]:
assert spark.version.startswith("3.5"), f"Unexpected version: {spark.version}"
assert spark.range(1).count() == 1
print("Connected to Spark", spark.version)

Connected to Spark 3.5.0-amzn-1


3 How Storage Works — Read vs Write
This is the most important change from Ocean Spark. In the old environment, everyone had folders
inside the shared AIS bucket and saved their work there. That is no longer how it works.
3.1 Two Locations, Two Roles

3.2 Accessing Your Personal Bucket in Code

In [3]:
import os
working_dir = os.environ["AWS_WORKING_DIRECTORY_PATH"]
# e.g. "datalab-142496269814-user-bucket/user-tkidpnkvdg/"

In [4]:
df = spark.read.parquet(
f"s3a://{working_dir}my_analysis/results.parquet"
)

3.3 Common Mistake: Writing to the AIS Bucket

In [5]:
# WRONG — the AIS bucket is read-only
user = "scharnou"
bucket = f"ais-data-142496269814/user_temp/{user}"
path = f"s3a://{bucket}/results/output.parquet"
df.write.mode("overwrite").parquet(path)
# → AccessDeniedException

AnalysisException: [PATH_NOT_FOUND] Path does not exist: s3a://datalab-142496269814-user-bucket/user-ewgcxn5mpw/my_analysis/results.parquet.

In [6]:
# RIGHT — write to your personal folder
import os
working_dir = os.environ["AWS_WORKING_DIRECTORY_PATH"]
path = f"s3a://{working_dir}results/output.parquet"
df.write.mode("overwrite").parquet(path)

AnalysisException: [PATH_NOT_FOUND] Path does not exist: s3a://datalab-142496269814-user-bucket/user-ewgcxn5mpw/my_analysis/results.parquet.

3.4 Summary

4 Reading AIS Data
4.1 Using af.get_ais() — The Recommended Way

In [52]:
from pyspark.sql import SparkSession
from ais import functions as af
from datetime import datetime
spark = SparkSession.builder.getOrCreate()
# One day, filtered by MMSI list
df = af.get_ais(
spark,
start_date=datetime(2024, 1, 15),
end_date=datetime(2024, 1, 15),
mmsi_list=[477555400, 538003913, 636092404],
)
df.show(5)

+------------+---------+-------------------+----------+----------+-------+-----------+--------+-----------+----------------+--------------------+------------+------+-----+----------------+---------+------------------+-------+-------+---+----+---+-------+----------+---------------+------+-------------------+-------------------+--------------------+--------------------+-------------------+--------------------+---------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+
|message_type|     mmsi|      dt_insert_utc| longitude|  latitude|    imo|vessel_name|callsign|vessel_type|vessel_type_code|   vessel_type_cargo|vessel_class|length|width|    flag_country|flag_code|       destination|    eta|draught|sog| cog|rot|heading|nav_status|nav

4.2 Multiple Specific Days (Non-Consecutive)

In [8]:
df = af.get_ais(
spark,
date_list=[datetime(2024, 1, 15), datetime(2024, 1, 16)],
mmsi_list=[477555400, 538003913],
)


4.3 Reading Raw Parquet Directly

In [11]:
import os
basepath = os.environ.get(
"AIS_BASEPATH",
"s3a://ais-data-142496269814/exact-earth-data/transformed/prod/",
)
# Single day
df = spark.read.parquet(f"{basepath}year=2024/month=01/day=15")
# Multiple days
df = spark.read.parquet(
f"{basepath}year=2024/month=01/day=15",
f"{basepath}year=2024/month=01/day=16",
)

4.4 Reading Ship Register CSVs

In [12]:
# List all available tables
tables = af.list_ihs_tables()
print(tables) # ['ShipData.CSV', 'tblShipTypeCodes.CSV', ...]
# Read a table
ship_types = af.read_ihs_table(spark, "tblShipTypeCodes.CSV")
ship_types.show(5)
# Read main ship data
ships = af.read_ihs_table(spark, "ShipData.CSV")
print(f"{ships.count():,} vessels in register")

['ShipData.CSV', 'tblAuxEngines.CSV', 'tblBoilers.CSV', 'tblBuilderDetails.CSV', 'tblCapacities.CSV', 'tblClassCodes.CSV', 'tblClassCurrent.CSV', 'tblClassHistory.CSV', 'tblCompanyFullDetails.CSV', 'tblCountryAndFlag.CSV', 'tblCrewList.CSV', 'tblFlagCodes.CSV', 'tblFlagHistory.CSV', 'tblGroupBeneficialOwnerHistory.CSV', 'tblMainEngines.CSV', 'tblNameHistory.CSV', 'tblOperatorHistory.CSV', 'tblPortOfRegistryFullCodes.CSV', 'tblPropulsionTypeDecode.CSV', 'tblRegisteredOwnerHistory.CSV', 'tblShipManagerHistory.CSV', 'tblShipTypeCodes.CSV', 'tblShipTypeHistory.CSV', 'tblSpecialFeatures.CSV', 'tblStatusCodes.CSV', 'tblStowageCommodity.CSV', 'version.csv']
+---------+--------------------+----------+--------------+----------+--------------+----------+--------------+------------------+--------------+--------------------+---------------+--------------------+
|StatCode5|      ShiptypeLevel5|Level4Code|ShipTypeLevel4|Level3Code|ShipTypeLevel3|Level2Code|ShipTypeLevel2|ShipTypeLevel1Code|ShiptypeL

In [13]:
ihs_basepath = "s3a://ais-data-142496269814/register/"
df = spark.read.csv(
f"{ihs_basepath}ShipData.CSV", header=True, inferSchema=True
)


4.5 Converting to Pandas


In [53]:
# Collect to pandas
pdf = df.toPandas()
# Collect to list of rows
rows = df.collect()

4.6 Data Volume Checks

In [15]:
from datetime import datetime, timedelta
today = datetime.now().date()
d = today - timedelta(days=1) # yesterday
path = f"{basepath}year={d.year}/month={d.month:02d}/day={d.day:02d}"
count = spark.read.parquet(path).count()
print(f"{d}: {count:,} rows")
assert count >= 20_000_000, f"Expected >= 20M rows, got {count:,}"

2026-04-07: 32,162,228 rows


5 Filtering Data
5.1 Broadcast Join with MMSI List

In [54]:
# Filter by a Python list
result = af.apply_small_filter(spark, df, [477555400, 538003913], "mmsi")
# Filter by a pandas DataFrame (extra columns are joined in)
import pandas as pd
targets = pd.DataFrame({
"mmsi": [477555400, 538003913],
"label": ["vessel_a", "vessel_b"],
})
result = af.apply_small_filter(spark, df, targets, "mmsi")
# result now has the "label" column attached

In [55]:
print(df.columns)

['message_type', 'mmsi', 'dt_insert_utc', 'longitude', 'latitude', 'imo', 'vessel_name', 'callsign', 'vessel_type', 'vessel_type_code', 'vessel_type_cargo', 'vessel_class', 'length', 'width', 'flag_country', 'flag_code', 'destination', 'eta', 'draught', 'sog', 'cog', 'rot', 'heading', 'nav_status', 'nav_status_code', 'source', 'dt_pos_utc', 'dt_static_utc', 'vessel_type_main', 'vessel_type_sub', 'eeid', 'source_filename', 'H3index_0', 'H3_int_index_0', 'H3_int_index_1', 'H3_int_index_2', 'H3_int_index_3', 'H3_int_index_4', 'H3_int_index_5', 'H3_int_index_6', 'H3_int_index_7', 'H3_int_index_8', 'H3_int_index_9', 'H3_int_index_10', 'H3_int_index_11', 'H3_int_index_12', 'H3_int_index_13', 'H3_int_index_14', 'H3_int_index_15']


5.2 H3 Hex Cell Filtering

In [57]:
# Singapore Strait H3 resolution-9 cells
h3_cells = [
618772217554534399, 618772216834424831, 618772216885805055
]
result = af.apply_small_filter(spark, df, h3_cells, "H3_int_index_9")
print(result)

DataFrame[H3_int_index_9: bigint, message_type: int, mmsi: int, dt_insert_utc: timestamp, longitude: double, latitude: double, imo: int, vessel_name: string, callsign: string, vessel_type: string, vessel_type_code: int, vessel_type_cargo: string, vessel_class: string, length: double, width: double, flag_country: string, flag_code: int, destination: string, eta: int, draught: double, sog: double, cog: double, rot: double, heading: double, nav_status: string, nav_status_code: int, source: string, dt_pos_utc: timestamp, dt_static_utc: timestamp, vessel_type_main: string, vessel_type_sub: string, eeid: bigint, source_filename: string, H3index_0: string, H3_int_index_0: bigint, H3_int_index_1: bigint, H3_int_index_2: bigint, H3_int_index_3: bigint, H3_int_index_4: bigint, H3_int_index_5: bigint, H3_int_index_6: bigint, H3_int_index_7: bigint, H3_int_index_8: bigint, H3_int_index_10: bigint, H3_int_index_11: bigint, H3_int_index_12: bigint, H3_int_index_13: bigint, H3_int_index_14: bigint, H

In [58]:
df = af.get_ais(
spark,
start_date=datetime(2024, 1, 15),
end_date=datetime(2024, 1, 15),
h3_list=[618772217554534399, 618772216834424831],
)


AttributeError: module 'h3.api.numpy_int' has no attribute 'h3_get_resolution'

5.3 Geo-Polygon Filtering with Sedona

In [19]:
polygon_wkt = (
"POLYGON((103.6 1.1, 104.1 1.1, "
"104.1 1.4, 103.6 1.4, 103.6 1.1))"
)
filtered = (
df
.filter(
~F.col("latitude").isNull()
& ~F.col("longitude").isNull()
)
.withColumn("_point",
F.expr("ST_Point(longitude, latitude)"))
.withColumn("_polygon",
F.expr(f"ST_GeomFromWKT('{polygon_wkt}')"))
.filter(F.expr("ST_Within(_point, _polygon)"))
.drop("_point", "_polygon")
)

In [59]:
polygon_geojson = {
"type": "Polygon",
"coordinates": [
[[103.6, 1.1], [104.1, 1.1],
[104.1, 1.4], [103.6, 1.4], [103.6, 1.1]]
],
}
df = af.get_ais(
spark,
start_date=datetime(2024, 1, 15),
end_date=datetime(2024, 1, 15),
polygon=polygon_geojson,
)

AttributeError: module 'h3.api.numpy_int' has no attribute 'polyfill'

5.4 Message Type Filtering

In [21]:
# Position reports only (types 1, 2, 3)
msg_types = [1, 2, 3]
positions = af.apply_small_filter(
spark, df, msg_types, "message_type"
)

Exception: column_filter should be in ais_df columns

5.5 Chaining Multiple Filters

In [22]:
# Start with a day of data
df = spark.read.parquet(
f"{basepath}year=2024/month=01/day=15"
)
# Filter by MMSI
mmsi_filtered = af.apply_small_filter(
spark, df, [477555400, 538003913], "mmsi"
)
# Then by message type
position_reports = af.apply_small_filter(
spark, mmsi_filtered, [1, 2, 3], "message_type"
)
# Then by H3 area
singapore = af.apply_small_filter(
spark, position_reports,
    [618772217554534399, 618772216834424831],
"H3_int_index_9",
)

6 Building DataFrames from Python Data
6.1 From a Pandas DataFrame (Recommended)


In [23]:
import pandas as pd
pdf = pd.DataFrame({
"mmsi": [477555400, 538003913, 636092404],
"vessel_name": ["EVER GIVEN", "PACIFIC VOYAGER", "CAPE TOWN"],
"sog": [12.5, 8.3, None],
})
df = spark.createDataFrame(pdf)

6.2 From a Python List or Dict


In [24]:
# List of dicts — simplest pattern
df = spark.createDataFrame([
{"mmsi": 477555400, "name": "EVER GIVEN", "sog": 12.5},
{"mmsi": 538003913, "name": "PACIFIC VOYAGER", "sog": 8.3},
])
# List of tuples with column names
df = spark.createDataFrame(
[(1, "Alice"), (2, "Bob")],
["id", "name"],
)


6.3 Geometry Columns (Sedona)

In [25]:
import shapely.geometry.base
def pandas_to_spark(spark, pdf):
"""Convert a pandas/geo DataFrame to Spark,
handling geometry columns."""
pdf = pdf.copy()
geom_cols = []
for col_name in pdf.columns:
if len(pdf) > 0 and isinstance(
pdf[col_name].iloc[0],
shapely.geometry.base.BaseGeometry,
):
geom_cols.append(str(col_name))
pdf[col_name] = pdf[col_name].apply(
lambda g: g.wkt if g is not None else None
)
df = spark.createDataFrame(pdf)
for col_name in geom_cols:
df = df.withColumn(
col_name,
F.expr(f"ST_GeomFromWKT({col_name})"),
)
return df


IndentationError: expected an indented block after function definition on line 2 (312044672.py, line 3)

6.4 Broadcast Join with Pandas

In [26]:
import pandas as pd
mmsi_pdf = pd.DataFrame({"mmsi": [477555400, 538003913, 636092404]})
mmsi_df = spark.createDataFrame(mmsi_pdf)
filtered = day_df.join(F.broadcast(mmsi_df), on="mmsi")


NameError: name 'day_df' is not defined

6.5 Legacy: F.array + F.lit Workaround

In [27]:
mmsi_list = [477555400, 538003913, 636092404]
arr = F.array(*[F.lit(v) for v in mmsi_list])
mmsi_df = spark.range(len(mmsi_list)).select(
F.element_at(
arr, (F.col("id") + 1).cast("int")
).alias("mmsi")
)

7 Aggregations and Window Functions
7.1 groupBy().agg() — Count, Min, Max, Mean


In [28]:
stats = df.groupBy("mmsi").agg(
F.count("*").alias("msg_count"),
F.min("sog").alias("min_sog"),
F.max("sog").alias("max_sog"),
F.mean("sog").alias("avg_sog"),
F.min("dt_pos_utc").alias("first_seen"),
F.max("dt_pos_utc").alias("last_seen"),
)
stats.show()

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `mmsi` cannot be resolved. Did you mean one of the following? [`id`, `name`].;
'Aggregate ['mmsi], ['mmsi, count(1) AS msg_count#4487L, 'min('sog) AS min_sog#4488, 'max('sog) AS max_sog#4489, 'avg('sog) AS avg_sog#4490, 'min('dt_pos_utc) AS first_seen#4491, 'max('dt_pos_utc) AS last_seen#4492]
+- Project [id#4479L AS id#4483L, name#4480 AS name#4484]
   +- LocalRelation [id#4479L, name#4480]


7.2 unionByName(allowMissingColumns=True)


In [29]:
merged = df1.unionByName(
df2, allowMissingColumns=True
)


NameError: name 'df1' is not defined

7.3 Window Functions: lag, row_number, first, last

In [30]:
from pyspark.sql.window import Window
w = Window.partitionBy("mmsi").orderBy("dt_pos_utc")
result = (
df
.withColumn("prev_sog", F.lag("sog").over(w))
.withColumn("rn", F.row_number().over(w))
)

In [31]:
w_unbounded = (
Window.partitionBy("mmsi")
.orderBy("dt_pos_utc")
.rowsBetween(
Window.unboundedPreceding,
Window.unboundedFollowing,
)
)
result = df.withColumn(
"first_sog", F.first("sog").over(w_unbounded)
)


7.4 F.expr() with SQL Window Strings

In [32]:
# Forward-fill nulls
result = df.withColumn(
"filled",
F.expr(
"coalesce(val, first(val, true) over ("
"partition by grp order by ord "
"rows between current row "
"and unbounded following"
"))"
),
)
# lag() via F.expr()
result = df.withColumn(
"prev_val",
F.expr(
"lag(val) over "
"(partition by grp order by ord)"
),
)

7.5 Route Assignment — af.assign_route()


In [33]:
result = af.assign_route(
df,
ship_unique_identifier_cols=["mmsi"],
route_order_by_cols=["dt_pos_utc"],
)
# Adds "route_group" column
# — increments when polygon_name changes

missing columns ['mmsi', 'dt_pos_utc', 'polygon_name']


7.6 Route Aggregation — af.agg_route()

In [34]:
route_stats = af.agg_route(
df,
group_by_cols=[
"mmsi", "route_group", "polygon_name"
],
order_by_cols=["dt_pos_utc"],
num_agg_cols=["sog", "draught"],
fl_agg_cols=["dt_pos_utc", "destination"],
checker=False,
)


missing columns ['mmsi', 'route_group', 'polygon_name', 'dt_pos_utc', 'sog', 'draught', 'dt_pos_utc', 'destination']


8 Sedona Spatial Functions
8.1 Available Functions

8.2 Always Use F.expr() — Not Client-Side Imports

In [35]:
# CORRECT — server-side registered function
df.withColumn(
"pt", F.expr("ST_Point(longitude, latitude)")
)
# WRONG — Sedona is not on the client pod
from sedona.sql import st_functions # FAILS

8.3 Full Geo-Filter Example

In [36]:
wkt = (
"POLYGON((103.6 1.1, 104.1 1.1, "
"104.1 1.4, 103.6 1.4, 103.6 1.1))"
)
filtered = (
df
.filter(
~F.col("latitude").isNull()
& ~F.col("longitude").isNull()
)
.withColumn("_point",
F.expr("ST_Point(longitude, latitude)"))
.withColumn("_polygon",
F.expr(f"ST_GeomFromWKT('{wkt}')"))
.filter(F.expr("ST_Within(_point, _polygon)"))
    .drop("_point", "_polygon")
)


8.5 createOrReplaceTempView + spark.sql() Alternative

In [37]:
df.createOrReplaceTempView("ais_data")
result = spark.sql(f"""
SELECT * FROM (
SELECT *,
ST_Point(longitude, latitude) as point,
ST_GeomFromWKT('{wkt}') as polygon
FROM ais_data
) WHERE ST_Within(point, polygon)
""")

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `longitude` cannot be resolved. Did you mean one of the following? [`id`, `name`].; line 4 pos 9;
'Project [*]
+- 'Filter 'ST_Within('point, 'polygon)
   +- 'SubqueryAlias __auto_generated_subquery_name
      +- 'Project [id#4764L, name#4765, 'ST_Point('longitude, 'latitude) AS point#4769,  **org.apache.spark.sql.sedona_sql.expressions.ST_GeomFromWKT**   AS polygon#4770]
         +- SubqueryAlias ais_data
            +- View (`ais_data`, [id#4764L,name#4765])
               +- Project [id#4760L AS id#4764L, name#4761 AS name#4765]
                  +- LocalRelation [id#4760L, name#4761]


8.6 SQL Joins via Temp Views

In [38]:
df1.createOrReplaceTempView("ships")
df2.createOrReplaceTempView("ports")
result = spark.sql("""
SELECT s.*, p.port_name
FROM ships s
INNER JOIN ports p ON s.port_id = p.id
""")


NameError: name 'df1' is not defined

In [39]:
# Equivalent DataFrame API (preferred)
result = df1.join(df2, on="port_id", how="inner")

NameError: name 'df1' is not defined

9 Writing Output Data
9.1 Your Personal S3 Working Directory

In [40]:
import os
working_dir = os.environ.get("AWS_WORKING_DIRECTORY_PATH")
# e.g. "datalab-142496269814-user-bucket/user-tkidpnkvdg/"

9.2 Write Parquet and Read It Back

In [41]:
output_path = (
f"s3a://{working_dir}my_analysis/results.parquet"
)
df.write.mode("overwrite").parquet(output_path)
# Read it back
df_check = spark.read.parquet(output_path)
print(f"Wrote and read back {df_check.count()} rows")

Wrote and read back 2 rows


9.3.1 File Explorer (No Code Required)
9.3.2 From a Notebook (S3 Commands)

In [42]:
import os
import boto3
working_dir = os.environ["AWS_WORKING_DIRECTORY_PATH"]
bucket = working_dir.split("/")[0]
prefix = "/".join(working_dir.split("/")[1:])
s3 = boto3.client("s3")
resp = s3.list_objects_v2(
Bucket=bucket,
Prefix=f"{prefix}ocean-spark-workspaces/",
Delimiter="/",
)
for cp in resp.get("CommonPrefixes", []):
print(cp["Prefix"].split("/")[-2])
import subprocess
ws_id = "wksp-561072-486d682a" # replace with yours
subprocess.run([
"aws", "s3", "cp", "--recursive",
f"s3://{working_dir}ocean-spark-workspaces/{ws_id}/",
f"./migrated-{ws_id}/",
])

IndentationError: expected an indented block after 'for' statement on line 12 (3777629871.py, line 13)

In [43]:
parquet_path = (
f"s3a://{working_dir}"
f"ocean-spark-workspaces/{ws_id}/my_results.parquet"
)
df = spark.read.parquet(parquet_path)


NameError: name 'ws_id' is not defined

9.4 Parse Bucket and Prefix (for boto3)

In [44]:
bucket_name = working_dir.split("/")[0]
# → "datalab-142496269814-user-bucket"
prefix = "/".join(working_dir.split("/")[1:])
# → "user-tkidpnkvdg/"

10 Ship Register Matching
10.1 af.match_ais_ihs() — Exact Matching


In [45]:
# Read ship register
ihs = af.read_ihs_table(spark, "ShipData.CSV")
# Match — ais_df needs: imo, mmsi, vessel_name
matched = af.match_ais_ihs(ais_df, ihs)

NameError: name 'ais_df' is not defined

10.2 Match Types

In [46]:
# Include unmatched rows
matched = af.match_ais_ihs(
ais_df, ihs, return_all=True
)

NameError: name 'ais_df' is not defined

10.3 Fuzzy Name Matching via @pandas_udf

In [47]:
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import DoubleType
from rapidfuzz import fuzz
@pandas_udf(DoubleType())
def name_similarity(
name1: pd.Series, name2: pd.Series
) -> pd.Series:
return pd.Series([
fuzz.ratio(n1, n2)
if pd.notna(n1) and pd.notna(n2)
else None
for n1, n2 in zip(name1, name2)
])
# Compare AIS vessel names against IHS register
result = df.withColumn(
"similarity",
name_similarity(
F.col("ais_name"), F.col("ihs_name")
),
)
# "EVER GIVEN" vs "EVER GIVEN" → 100.0
# "EVERGIVEN" vs "EVER GIVEN" → ~89.5


IndentationError: expected an indented block after function definition on line 5 (159793626.py, line 8)

11 What Does NOT Work

11.1 spark.sparkContext


In [48]:
# FAILS:
spark.sparkContext.setLogLevel("ERROR")


PySparkNotImplementedError: [NOT_IMPLEMENTED] sparkContext() is not implemented.

11.2 F.udf() — Works for Simple Cases

In [49]:
from pyspark.sql.types import StringType
upper_udf = F.udf(lambda x: x.upper(), StringType())
result = df.withColumn("name_upper", upper_udf(F.col("vessel_name")))

11.3 spark.stop() — Never Call It


In [50]:
# WRONG — breaks your session permanently
#spark.stop()
# WRONG — same problem, different syntax
#spark.client.close()
# RIGHT — just close the notebook tab and stop the
# service from the Onyxia UI when you are done

11.4 createDataFrame() from Python Lists (Fixed)
11.5 Client-Side Sedona Imports